# Laboratorio: Su primera red neuronal — Predicción de compra en retail

<p style="text-align:center">
Universidad del Valle<br>
Deep Learning 2026<br>
Kevin Recinos
</p>

---

**Contexto.** Una tienda en línea quiere predecir si un cliente que está navegando el sitio va a **completar la compra** (checkout) o no, usando dos señales ya normalizadas en [0, 1]:

- `tiempo_en_sitio`: tiempo que el cliente ha pasado navegando, normalizado.
- `valor_carrito`: valor de lo que lleva en el carrito, normalizado.

**Arquitectura.** Esta es la misma red que hemos trabajado en clase con notación matricial: 2 entradas → 1 capa oculta con 2 neuronas (ReLU) → 1 neurona de salida (sigmoid) → pérdida BCE.

$$
z^{(1)} = W^{(1)}x + b^{(1)}, \quad a^{(1)} = \text{ReLU}(z^{(1)})
$$
$$
z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}, \quad \hat{y} = \sigma(z^{(2)})
$$

**Cómo se organiza este laboratorio.** No todos los bloques piden lo mismo — es importante que lo tenga claro desde el inicio:

| Bloques | Qué va a hacer |
|---|---|
| 0b | **Investigar.** Preguntas cortas sobre NumPy, con una lectura guiada — antes de escribir código. |
| 1–6 | **Desarrollar código.** Usted escribe, con NumPy, el forward pass, la pérdida, el backward pass y un paso de actualización — cerrando la parte matemática que hemos construido en clase. |
| 7 | **Investigar y luego ejecutar.** Primero responde unas preguntas cortas sobre PyTorch (con una lectura guiada). Después *ejecuta* código ya construido — no lo desarrolla — y compara sus resultados. |
| 8 | Preguntas de análisis, en sus propias palabras. |
| 9 | Conclusión: 2 preguntas de reflexión personal. |

Cada bloque de código de las secciones 1–6 incluye un mecanismo de autoverificación (`assert` con un hash o una tolerancia numérica). Si su bloque imprime `OK`, el resultado es correcto. Si el `assert` falla, revise su implementación antes de continuar — los bloques posteriores dependen de los anteriores.

**Entrega.** Todo su trabajo —código, respuestas de investigación, análisis y conclusión— va dentro de este mismo notebook, en celdas de Markdown donde corresponda. **Lo único que debe subir es este archivo `.ipynb`**, con todas las celdas ejecutadas.


## Bloque 0: Configuración

Ejecute esta celda sin modificarla. Define los datos del ejemplo (un cliente), los pesos iniciales de la red (sin entrenar) y las funciones de verificación.


In [ ]:
import numpy as np
import hashlib

def _hash(arr, decimals=6):
    return hashlib.sha256(np.round(np.array(arr, dtype=float), decimals).tobytes()).hexdigest()

HASHES = {
    "z1":        "cafad3f1ed81add62c422900bd127600c4eb5dc04006fb4c9ec0164ebaa6d2e1",
    "a1":        "cafad3f1ed81add62c422900bd127600c4eb5dc04006fb4c9ec0164ebaa6d2e1",
    "z2":        "ec1a3ac855a4975f6bf6b375a45b3f351160740545346ac11b81659bc0f4bdf1",
    "y_hat":     "7e374e66acdc7a40ec248f4686a8545282982743448fada4714d95fdc8c033ce",
    "dL_dz2":    "c11f69f948cc6824d04a19d686d912c157c2b126bb3d628695060a648d567064",
    "dL_dW2":    "67aca37e8a3a82676df0de3749fd329c90291cf4b94ed4b04ba3a71cad26af63",
    "dL_db2":    "c11f69f948cc6824d04a19d686d912c157c2b126bb3d628695060a648d567064",
    "dL_da1":    "e57f3b2ac85328af556b68d8c6a32173091031b89e4264f137ceb04070d7ad6a",
    "dL_dz1":    "e57f3b2ac85328af556b68d8c6a32173091031b89e4264f137ceb04070d7ad6a",
    "dL_dW1":    "6400a4e794eb078da32acef911c22d2028c5a6ddd42166a4e736fb052ff1f247",
    "dL_db1":    "e57f3b2ac85328af556b68d8c6a32173091031b89e4264f137ceb04070d7ad6a",
    "W1_new":    "ddb9540386be29e24f29a2aa3424fe4a2537efa440164d47b8105fafa3b9aee5",
    "b1_new":    "dbc3812a9386d57e7044ef77ec3a7e4dbb01d7861504fe9221a51ff4881d0e67",
    "W2_new":    "7698b8f8210035cad55d027e80c636ed48cdfdc033a6f21eeba8f06b9b94b643",
    "b2_new":    "2ddc627e1cc8545f8f326858c19683fd8e628f0c9316b317bea765ddfb48b585",
}

# Cliente de e-commerce retail: dos variables ya normalizadas en [0,1]
x = np.array([0.5, 0.8])   # [tiempo_en_sitio_norm, valor_carrito_norm]
y = 1.0                    # 1 = el cliente completo la compra (checkout)

W1 = np.array([[0.3, -0.2],
               [0.5,  0.4]])   # (2,2): fila = neurona oculta, columna = entrada
b1 = np.array([0.1, -0.1])     # (2,)

W2 = np.array([[0.6, -0.3]])   # (1,2)
b2 = np.array([0.2])           # (1,)

alpha = 0.4

print("Setup listo.")


---
## Bloque 0b: Investigación — NumPy

Antes de escribir código, si no ha trabajado antes con NumPy (o si necesita repasar), lea la guía oficial **"NumPy: the absolute basics for beginners"**:

<https://numpy.org/doc/stable/user/absolute_beginners.html>

No necesita leerla completa: con la introducción a los arrays, la sección de operaciones básicas y la de "forma" (shape) es suficiente para este laboratorio. Responda, en una celda de Markdown debajo de esta, con sus propias palabras:

1. ¿Qué es un array de NumPy, y en qué se diferencia de una lista de Python?
2. ¿Qué representa la "forma" (`shape`) de un array? Dé un ejemplo con un array de forma `(2,2)` y uno de forma `(2,)`.
3. ¿Qué diferencia hay entre el operador `@` y el operador `*` entre dos arrays?
4. ¿Qué es "broadcasting" en NumPy? (esto le va a ayudar a entender por qué se puede sumar un vector de sesgos a una matriz sin errores de forma)


*(Escriba aqui sus respuestas a las 4 preguntas del Bloque 0b)*

---
## Bloque 1: Forward — capa oculta

Implemente:
- `relu(z)`: función de activación ReLU.
- `z1`: preactivación de la capa oculta, $z^{(1)} = W^{(1)}x + b^{(1)}$.
- `a1`: activación de la capa oculta, $a^{(1)} = \text{ReLU}(z^{(1)})$.

**Pista:** `W1` tiene forma (2,2) y `x` tiene forma (2,) — la multiplicación matricial correcta es `W1 @ x`, no `x @ W1` (en este caso ambas producen la misma forma de salida, pero solo una es la operación correcta; piense en qué representa cada fila de `W1`).


In [ ]:
def relu(z):
    # TODO: implemente ReLU
    pass

z1 = None  # TODO: z1 = W1 @ x + b1
a1 = None  # TODO: a1 = relu(z1)

assert _hash(z1) == HASHES["z1"], "Hash de z1 no coincide."
assert _hash(a1) == HASHES["a1"], "Hash de a1 no coincide."
print("Bloque 1: OK")
print(f"z1 = {np.round(z1, 6)}")
print(f"a1 = {np.round(a1, 6)}")


---
## Bloque 2: Forward — capa de salida

Implemente:
- `sigmoid(z)`.
- `z2`: preactivación de salida, $z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}$.
- `y_hat`: la predicción final, como **float escalar** (no como arreglo de una posición).


In [ ]:
def sigmoid(z):
    # TODO: implemente sigmoid
    pass

z2 = None      # TODO
y_hat = None   # TODO: recuerde: float escalar, no arreglo

assert _hash(z2) == HASHES["z2"], "Hash de z2 no coincide."
assert _hash([y_hat]) == HASHES["y_hat"], "Hash de y_hat no coincide."
print("Bloque 2: OK")
print(f"z2 = {np.round(z2, 6)}")
print(f"y_hat = {y_hat:.6f}")


---
## Bloque 3: Pérdida (Binary Cross-Entropy)

Implemente `bce_loss(y_hat, y)`. Recuerde agregar un `eps = 1e-8` dentro de los logaritmos para evitar `log(0)`.


In [ ]:
def bce_loss(y_hat, y):
    eps = 1e-8
    # TODO: implemente la formula de BCE
    pass

loss = None  # TODO
assert abs(loss - 0.6382424381456336) < 1e-5, f"Loss incorrecto: {loss:.6f}"
print("Bloque 3: OK")
print(f"loss = {loss:.6f}")


---
## Bloque 4: Backward — capa de salida

Implemente, usando la simplificación de la derivada de BCE + sigmoid que vimos en clase ($\partial L/\partial z^{(2)} = \hat{y} - y$):

- `dL_dz2`
- `dL_dW2` (forma (1,2))
- `dL_db2` (forma (1,))


In [ ]:
dL_dz2 = None  # TODO: use la simplificacion BCE+sigmoid
dL_dW2 = None  # TODO: forma (1,2)
dL_db2 = None  # TODO: forma (1,)

assert _hash([dL_dz2]) == HASHES["dL_dz2"], "Hash de dL_dz2 no coincide."
assert _hash(dL_dW2) == HASHES["dL_dW2"], "Hash de dL_dW2 no coincide."
assert _hash(dL_db2) == HASHES["dL_db2"], "Hash de dL_db2 no coincide."
print("Bloque 4: OK")
print(f"dL_dz2 = {dL_dz2:.6f}")
print(f"dL_dW2 = {np.round(dL_dW2, 6)}")
print(f"dL_db2 = {np.round(dL_db2, 6)}")


---
## Bloque 5: Backward — capa oculta

Aplique la regla de la cadena hasta la capa oculta. Implemente:

- `relu_deriv(z)`: 1 donde `z > 0`, 0 en otro caso.
- `dL_da1`: gradiente de la pérdida respecto a `a1` (use `dL_dz2` y `W2`).
- `dL_dz1`: `dL_da1` multiplicado elemento a elemento por `relu_deriv(z1)`.
- `dL_dW1`: use `np.outer(dL_dz1, x)` (forma (2,2)).
- `dL_db1`.

**Pregúntese antes de escribir código:** ¿por qué `dL_dz1` necesita multiplicarse por la derivada de ReLU y `dL_da1` no?


In [ ]:
def relu_deriv(z):
    # TODO
    pass

dL_da1 = None  # TODO
dL_dz1 = None  # TODO
dL_dW1 = None  # TODO: use np.outer
dL_db1 = None  # TODO

assert _hash(dL_da1) == HASHES["dL_da1"], "Hash de dL_da1 no coincide."
assert _hash(dL_dz1) == HASHES["dL_dz1"], "Hash de dL_dz1 no coincide."
assert _hash(dL_dW1) == HASHES["dL_dW1"], "Hash de dL_dW1 no coincide."
assert _hash(dL_db1) == HASHES["dL_db1"], "Hash de dL_db1 no coincide."
print("Bloque 5: OK")
print(f"dL_dz1 = {np.round(dL_dz1, 6)}")
print(f"dL_dW1 =\n{np.round(dL_dW1, 6)}")
print(f"dL_db1 = {np.round(dL_db1, 6)}")


---
## Bloque 6: Un paso de descenso de gradiente

Actualice los cuatro parámetros (`W1`, `b1`, `W2`, `b2`) usando `param_new = param - alpha * gradiente`. Luego recalcule el forward pass completo con los parámetros nuevos (`z1_new`, `a1_new`, `z2_new`, `y_hat_new`, `loss_new`) para verificar que la pérdida bajó.


In [ ]:
W1_new = None  # TODO
b1_new = None  # TODO
W2_new = None  # TODO
b2_new = None  # TODO

z1_new = None      # TODO: recalcule el forward completo
a1_new = None
z2_new = None
y_hat_new = None
loss_new = None

assert _hash(W1_new) == HASHES["W1_new"], "Hash de W1_new no coincide."
assert _hash(b1_new) == HASHES["b1_new"], "Hash de b1_new no coincide."
assert _hash(W2_new) == HASHES["W2_new"], "Hash de W2_new no coincide."
assert _hash(b2_new) == HASHES["b2_new"], "Hash de b2_new no coincide."
assert loss_new < loss, f"La perdida no bajo: {loss:.6f} -> {loss_new:.6f}"
print("Bloque 6: OK")
print(f"loss antes:   {loss:.6f}")
print(f"loss despues: {loss_new:.6f}")
print(f"y_hat antes:   {y_hat:.6f}")
print(f"y_hat despues: {y_hat_new:.6f}")


---
## Bloque 7a: Investigación — PyTorch y autograd

Antes de tocar código de PyTorch, lea la sección **"A Gentle Introduction to torch.autograd"** de la documentación oficial:

<https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html>

No necesita leerla completa ni entender cada detalle — con la introducción y la sección de uso básico es suficiente para responder lo siguiente. Escriba sus respuestas en una celda de Markdown debajo de esta, con sus propias palabras:

1. ¿Qué es un tensor en PyTorch, y en qué se parece o se diferencia de un array de NumPy que ya usó en los Bloques 1–6?
2. ¿Qué significa marcar un tensor con `requires_grad=True`? ¿Qué cambiaría si no lo marca?
3. Cuando se llama `.backward()` sobre la pérdida, ¿qué hace PyTorch internamente, en términos generales? (relacione esto con la regla de la cadena que usted aplicó a mano en los Bloques 4 y 5)
4. Después de llamar `.backward()`, ¿en qué atributo del tensor queda guardado su gradiente?


*(Escriba aqui sus respuestas a las 4 preguntas del Bloque 7a)*

---
## Bloque 7b: Ejecución guiada — verificar los gradientes con autograd

Ahora sí, a diferencia de los Bloques 1–6, **no va a desarrollar este código.** Ya está completo. Su tarea es ejecutarlo, leerlo línea por línea apoyándose en lo que investigó en el Bloque 7a, y luego responder la reflexión que sigue.

**Al ejecutar la celda, compare la salida contra los gradientes `dL_dW1`, `dL_db1`, `dL_dW2`, `dL_db2` que usted calculó a mano en los Bloques 4 y 5.**

*(Reflexión — respóndala en una celda de Markdown debajo del código): ¿coincidieron los cuatro gradientes? Señale, línea por línea del código, en qué momento PyTorch hace lo que usted hizo manualmente en el Bloque 5 (la regla de la cadena hacia la capa oculta).*


In [ ]:
import torch

# Los mismos parametros y datos, pero como tensores de PyTorch.
# Esta celda ya esta completa: ejecutela y lea con calma cada linea,
# apoyandose en lo que investigo en el Bloque 7a.
W1_t = torch.tensor(W1, requires_grad=True)
b1_t = torch.tensor(b1, requires_grad=True)
W2_t = torch.tensor(W2, requires_grad=True)
b2_t = torch.tensor(b2, requires_grad=True)
x_t  = torch.tensor(x)
y_t  = torch.tensor(y)

z1_t = W1_t @ x_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = W2_t @ a1_t + b2_t
y_hat_t = torch.sigmoid(z2_t[0])

loss_t = -(y_t * torch.log(y_hat_t + 1e-8) + (1 - y_t) * torch.log(1 - y_hat_t + 1e-8))
loss_t.backward()   # <- aqui ocurre lo que investigo en el Bloque 7a

print("Comparacion gradiente manual (Bloques 4-5) vs PyTorch autograd:\n")
for nombre, manual, auto in [
    ("dL_dW1", dL_dW1, W1_t.grad.numpy()),
    ("dL_db1", dL_db1, b1_t.grad.numpy()),
    ("dL_dW2", dL_dW2, W2_t.grad.numpy()),
    ("dL_db2", dL_db2, b2_t.grad.numpy()),
]:
    coincide = np.allclose(manual, auto, atol=1e-6)
    diff = np.max(np.abs(manual - auto))
    print(f"{nombre:8s} | coincide: {str(coincide):5s} | diferencia maxima: {diff:.2e}")


*(Escriba aqui su reflexion del Bloque 7b)*

---
## Bloque 8: Preguntas de análisis

Responda en celdas de texto (Markdown) debajo de cada pregunta. No se espera una redacción extensa: se evalúa la calidad del argumento, no la longitud.

**Pregunta 1.** Antes de que la red viera este cliente, sus pesos ya tenían los valores fijos que le dimos en el Bloque 0 — no fueron entrenados con datos reales de la tienda. ¿Qué le dice esto sobre qué tan confiable es `y_hat` en este punto del laboratorio?

**Pregunta 2.** Después de un solo paso de descenso de gradiente, la pérdida bajó (Bloque 6). ¿Significa esto que la red ya clasificaría correctamente a este cliente como "sí va a comprar"? Justifique con el valor de `y_hat_new` y el umbral de decisión de 0.5.

**Pregunta 3.** Observe el signo de cada valor en `dL_dW2`. ¿Qué le indica el signo sobre la dirección en la que se actualizará cada peso, y por qué la actualización usa `- alpha * gradiente` y no `+ alpha * gradiente`?

**Pregunta 4.** En este ejemplo, ambas neuronas de la capa oculta terminaron con `z1 > 0`, así que ReLU no "apagó" a ninguna. Si una neurona oculta tuviera `z1 < 0`, ¿qué le pasaría a su gradiente durante el backward (Bloque 5), y qué riesgo de entrenamiento se conecta con esto?

**Pregunta 5 (opinión personal).** Ahora que implementó el mismo cálculo dos veces —a mano con NumPy y con autograd de PyTorch— ¿qué ventaja y qué desventaja le ve usted a calcular los gradientes manualmente, considerando lo que le costó llegar al Bloque 7?


*(Escriba su respuesta a la Pregunta 1 aqui)*

*(Escriba su respuesta a la Pregunta 2 aqui)*

*(Escriba su respuesta a la Pregunta 3 aqui)*

*(Escriba su respuesta a la Pregunta 4 aqui)*

*(Escriba su respuesta a la Pregunta 5 aqui)*

---
## Bloque 9: Conclusión

Responda, en una celda de Markdown debajo de esta, con sus propias palabras. No hace falta que sea largo, pero sí que sea suyo.

**1.** De todo el laboratorio —investigar NumPy, desarrollar el forward y el backward a mano, investigar PyTorch, y ejecutar la verificación con autograd— ¿qué parte le costó más, y qué fue específicamente lo que se le dificultó?

**2.** Ahora que implementó el mismo cálculo dos veces —a mano y con autograd— ¿qué ventaja y qué desventaja le encuentra usted a calcular los gradientes manualmente, considerando lo que le costó llegar hasta el Bloque 7?


*(Escriba su respuesta a la Pregunta 1 de la Conclusion aqui)*

*(Escriba su respuesta a la Pregunta 2 de la Conclusion aqui)*